# Scraping Top Repositories in Github Topics

### 
- We will scrape https://github.com/topics.
- We will get the list of topics. For each topic, we'll get the topic title, topic page URL and topic description.
- We will get the top repos in a given topic, the owner's name, its URL and the number of stars it has.

### I used the request library to download the web page

In [106]:
%pip install requests --upgrade --quiet

Note: you may need to restart the kernel to use updated packages.


In [107]:
import requests

In [108]:
topics_url = "https://github.com/topics"
response = requests.get(topics_url)

In [109]:
response.status_code

200

In [110]:
with open('webpage.html','w') as f:
    f.write(response.text)

### I used Beautiful Soup to parse and extract information

In [111]:
%pip install beautifulsoup4 --upgrade --quiet

Note: you may need to restart the kernel to use updated packages.


In [112]:
from bs4 import BeautifulSoup
doc = BeautifulSoup(response.text, 'html.parser')

In [113]:
topic_title_selection_class = "f3 lh-condensed mb-0 mt-1 Link--primary"
topic_desc_selection_class = "f5 color-fg-muted mb-0 mt-1"
topic_link_selection_class = "no-underline flex-1 d-flex flex-column"

topic_title_tags = doc.find_all("p",{"class": topic_title_selection_class})

topic_desc_tag = doc.find_all("p",{"class": topic_desc_selection_class })

topic_link_tag = doc.find_all("a",{"class": topic_link_selection_class })


In [114]:
topic_titles = []

for tag in topic_title_tags:
    topic_titles.append(tag.text)

In [115]:
topic_descs = []

for desc in topic_desc_tag:
    topic_descs.append((desc.text).strip())

In [116]:
topic_links = []

for link in topic_link_tag:
    topic_links.append('https://github.com' + link['href'])

In [117]:
%pip install pandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [118]:
import pandas as pd 

In [119]:
dict = {'Title' : topic_titles,
        'Description' : topic_descs,
        'URL' : topic_links
        }

topic_df = pd.DataFrame(dict)

### Created CSV files with extracted information

In [120]:
topic_df.to_csv('topics.csv', index = None)

### Getting information out of a topic page in a CSV file

In [128]:
for topic_page_link in topic_links:

    username = []
    repo_name = []
    repo_url = []
    stars = []

    response  = requests.get(topic_page_link)
    topic_doc = BeautifulSoup(response.text, 'html.parser')

    h3_selection_class = "f3 color-fg-muted text-normal lh-condensed"

    repo_tags = topic_doc.find_all("h3", {"class": h3_selection_class}) 

    star_selection_class = "tooltipped tooltipped-sw btn-sm btn color-bg-default"

    star_tags = topic_doc.find_all("a",{"class":star_selection_class})

    for repo_tag in repo_tags:
        a_tags = repo_tag.find_all("a")

        username.append(a_tags[0].text.strip())
        repo_name.append(a_tags[1].text.strip())
        repo_url.append("https://github.com" + a_tags[1]["href"])

    for star_tag in star_tags:
        star_count_tag = star_tag.find_all("span")
        stars.append(star_count_tag[1].text.strip())

    dict = {"Username":username,
            "Repository Name":repo_name,
            "Stars":stars,
            "Repository URL":repo_url}
    
    topic_df = pd.DataFrame(dict)
    topic = topic_page_link[26:]
    topic_df.to_csv(topic+'.csv', index = None)